# Thread Pools — Multilenguaje (Python, C++, Java, C#, Rust)

**Objetivo:** reutilizar hilos para muchas tareas pequeñas.
- Beneficio: reducir *overhead* de crear/destruir hilos.
- **I/O-bound**: buenos resultados con pools.
- **CPU-bound (CPython)**: GIL impide paralelismo real → usar `ProcessPool` o extensiones nativas.
- **Paralelismo real** depende del SO y los núcleos.


## 0) Instalación (si no corriste antes)

In [1]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq g++ openjdk-17-jdk mono-devel
command -v rustc >/dev/null 2>&1 || (curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y)
source $HOME/.cargo/env || true
echo 'Listo.'


Selecting previously unselected package mono-runtime-common.
(Reading database ... 126435 files and directories currently installed.)
Preparing to unpack .../0-mono-runtime-common_6.8.0.105+dfsg-3.2_amd64.deb ...
Unpacking mono-runtime-common (6.8.0.105+dfsg-3.2) ...
Selecting previously unselected package libmono-corlib4.5-dll.
Preparing to unpack .../1-libmono-corlib4.5-dll_6.8.0.105+dfsg-3.2_all.deb ...
Unpacking libmono-corlib4.5-dll (6.8.0.105+dfsg-3.2) ...
Selecting previously unselected package libmono-system-core4.0-cil.
Preparing to unpack .../2-libmono-system-core4.0-cil_6.8.0.105+dfsg-3.2_all.deb ...
Unpacking libmono-system-core4.0-cil (6.8.0.105+dfsg-3.2) ...
Selecting previously unselected package libmono-system-numerics4.0-cil.
Preparing to unpack .../3-libmono-system-numerics4.0-cil_6.8.0.105+dfsg-3.2_all.deb ...
Unpacking libmono-system-numerics4.0-cil (6.8.0.105+dfsg-3.2) ...
Selecting previously unselected package libmono-system-xml4.0-cil.
Preparing to unpack .../4-

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
info: downloading installer
info: profile set to 'default'
info: default host triple is x86_64-unknown-linux-gnu
info: syncing channel updates for 'stable-x86_64-unknown-linux-gnu'
info: latest update on 2025-08-07, rust version 1.89.0 (29483883e 2025-08-04)
info: downloading component 'cargo'
info: downloading component 'clippy'
info: downloading component 'rust-docs'
info: downloading component 'rust-std'
info: downloading component 'rustc'
info: downloading component 'rustfmt'
info: installing component 'cargo'
info: installing component 'clippy'
info: installing component 'rust-docs'
info: installing component 'rust-std'
info: installing component 'rustc'
info: installing component 'rustfmt'
info: default toolchain set to 'stable-x86_64-unknown-linux-gnu'


## 1) Python — `ThreadPoolExecutor` vs threads manuales

In [2]:
import time, threading, concurrent.futures as cf
N=60; SLEEP=0.03

def tiny_io(i): time.sleep(SLEEP); return i*i

# Threads manuales
ths=[threading.Thread(target=tiny_io,args=(i,)) for i in range(N)]
t0=time.perf_counter()
[t.start() for t in ths]; [t.join() for t in ths]
print("Threads manuales:", time.perf_counter()-t0, "seg")

# Pool
t0=time.perf_counter()
with cf.ThreadPoolExecutor(max_workers=20) as ex:
    list(ex.map(tiny_io, range(N)))
print("ThreadPool:", time.perf_counter()-t0, "seg")

Threads manuales: 0.038627636000001075 seg
ThreadPool: 0.09555487699998366 seg


## 2) C++ — Pool simple (cola + workers)

In [3]:
%%writefile pool.cpp
#include <bits/stdc++.h>
#include <thread>
#include <queue>
#include <functional>
#include <condition_variable>
using namespace std;

class ThreadPool
{
  vector<thread> workers;
  queue<function<void()>> tasks;
  mutex m; condition_variable cv; bool stop=false;
 public:
  ThreadPool(size_t n)
  {
    for(size_t i=0;i<n;i++)
    {
      workers.emplace_back([this]
      {
        while(true)
        {
          function<void()> task;
          {
            unique_lock<mutex> lk(m);
            cv.wait(lk,[this]{ return stop || !tasks.empty(); });
            if(stop && tasks.empty()) return;
            task = move(tasks.front()); tasks.pop();
          }
          task();
        }
      });
    }
  }
  template<class F> void submit(F f)
  {
    { lock_guard<mutex> lk(m); tasks.emplace(f); }
    cv.notify_one();
  }
  ~ThreadPool()
  {
    { lock_guard<mutex> lk(m); stop=true; }
    cv.notify_all();
    for(auto& w:workers) w.join();
  }
};

int main()
{
  const int N=60; const int WORKERS=8;
  ThreadPool pool(WORKERS);
  auto t0 = chrono::steady_clock::now();
  mutex out;
  atomic<int> done=0;
  for(int i=0;i<N;i++)
  {
    pool.submit([i,&out,&done]
    {
      this_thread::sleep_for(chrono::milliseconds(30)); // I/O sim
      int r = i*i;
      lock_guard<mutex> lk(out);
      (void)r; done++;
    });
  }
  while(done<N) this_thread::sleep_for(chrono::milliseconds(1));
  auto dt = chrono::duration<double>(chrono::steady_clock::now()-t0).count();
  cout << "Pool C++ tiempo: " << dt << " s\n";
}


Writing pool.cpp


In [4]:
!g++ -O2 -lpthread pool.cpp -o pool

In [5]:
!time ./pool

Pool C++ tiempo: 0.242006 s

real	0m0.246s
user	0m0.005s
sys	0m0.003s


In [6]:
!ls -la

total 48
drwxr-xr-x 1 root root  4096 Sep 17 15:24 .
drwxr-xr-x 1 root root  4096 Sep 17 15:20 ..
drwxr-xr-x 4 root root  4096 Sep 15 17:50 .config
-rwxr-xr-x 1 root root 28664 Sep 17 15:24 pool
-rw-r--r-- 1 root root  1524 Sep 17 15:24 pool.cpp
drwxr-xr-x 1 root root  4096 Sep 15 17:50 sample_data


## 3) Java — `Executors.newFixedThreadPool`

In [7]:
%%writefile Pool.java
import java.util.concurrent.*;
public class Pool
{
  static void sleep(long ms){ try{ Thread.sleep(ms);}catch(Exception e){} }
  public static void main(String[] a)
  {
    ExecutorService ex = Executors.newFixedThreadPool(8);
    long t0 = System.currentTimeMillis();
    for(int i=0;i<60;i++)
    {
      final int n=i;
      ex.submit(() -> { sleep(30); return n*n; });
    }
    ex.shutdown();
    try { ex.awaitTermination(10, TimeUnit.SECONDS); } catch(Exception e){}
    System.out.println("Tiempo pool Java: "+(System.currentTimeMillis()-t0)+" ms");
  }
}


Writing Pool.java


In [8]:
!javac Pool.java
!time java Pool

Tiempo pool Java: 253 ms

real	0m0.320s
user	0m0.083s
sys	0m0.026s


## 4) C# — `Parallel.For` / ThreadPool

In [16]:
%%writefile Pool.cs
using System;
using System.Threading.Tasks;
using System.Diagnostics;
class P
{
  static void Sleep(int ms){ System.Threading.Thread.Sleep(ms); }
  static void Main()
  {
    var sw = Stopwatch.StartNew();
    Parallel.For(0, 60, i => { Sleep(30); var r = i*i; });
    sw.Stop();
    Console.WriteLine("Tiempo pool C#: " + sw.ElapsedMilliseconds + " ms");
  }
}


Overwriting Pool.cs


In [17]:
!mcs -langversion:latest Pool.cs -out:pool.exe
!time mono pool.exe

Pool.cs(10,47): warning CS0219: The variable `r' is assigned but its value is never used
Compilation succeeded - 1 warning(s)
Tiempo pool C#: 917 ms

real	0m0.947s
user	0m0.037s
sys	0m0.011s


## 5) Rust — `rayon` (paralelismo de datos)

In [20]:
%%bash
set -e
source $HOME/.cargo/env
cargo new --bin pool_rayon >/dev/null 2>&1 || true
cd pool_rayon
cat > Cargo.toml << 'TOML'
[package]
name = "pool_rayon"
version = "0.1.0"
edition = "2021"

[dependencies]
rayon = "1.8"
TOML
cat > src/main.rs << 'RS'
use rayon::prelude::*;
use std::time::Instant;
use std::thread;
use std::time::Duration;

fn main()
{
    let v: Vec<i32> = (0..60).collect();
    let t0 = Instant::now();
    let r: Vec<i32> = v.par_iter().map(|x| { thread::sleep(Duration::from_millis(30)); x*x }).collect();
    println!("n={} tiempo={:?}", r.len(), t0.elapsed());
}
RS
time cargo run --quiet

n=60 tiempo=905.146286ms



real	0m1.338s
user	0m0.309s
sys	0m0.131s


In [19]:
!ls -la

total 68
drwxr-xr-x 1 root root  4096 Sep 17 15:26 .
drwxr-xr-x 1 root root  4096 Sep 17 15:20 ..
drwxr-xr-x 4 root root  4096 Sep 15 17:50 .config
-rwxr-xr-x 1 root root 28664 Sep 17 15:24 pool
-rw-r--r-- 1 root root  2180 Sep 17 15:24 Pool.class
-rw-r--r-- 1 root root  1524 Sep 17 15:24 pool.cpp
-rw-r--r-- 1 root root   362 Sep 17 15:26 Pool.cs
-rwxr-xr-x 1 root root  3584 Sep 17 15:26 pool.exe
-rw-r--r-- 1 root root   564 Sep 17 15:24 Pool.java
drwxr-xr-x 5 root root  4096 Sep 17 15:26 pool_rayon
drwxr-xr-x 1 root root  4096 Sep 15 17:50 sample_data


### Conclusión
- El *thread pool* mejora throughput en **I/O-bound** y en CPUs con múltiples núcleos.
- En **CPython**, para **CPU-bound** usar `ProcessPool` (paralelismo real) o librerías nativas.
- El rendimiento final **depende del SO y hardware**.
